# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row** = one content item, for one client, on one calendar day — grain
`(client_hash_id, content_hash_id, report_date)`, from `fact_content_daily_performance`.

**Time window (this notebook):** iterating on the mid-panel month `month = '2026-03'` — a full
calendar month, away from both panel edges. The full release spans `2025-01-27` to `2026-06-30`
(~17 months); June 2026 (`fact_content_daily_performance_sample`) is the panel's final month and
is sealed as a test month — never used here to develop label logic, per the warning on this card.

**Tables used:** `fact_content_daily_performance` (main, daily grain), `dim_clients` (panel
depth / access flags), `dim_content` (content-level metadata). `fact_content_query_90d` is
noted but not used here — see "excluded" below.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
# Token order: env var -> Colab Secret -> prompt (last resort). Store as a Colab Secret
# named HF_TOKEN so the prompt never fires — never paste a token into a cell, this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = "2026-03"  # mid-panel month — iterate here, never on the sealed final month (2026-06)

window = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
print("Unit of analysis: one row = one (client_hash_id, content_hash_id, report_date) in")
print(f"fact_content_daily_performance, filtered to month = {MONTH}.")
window


Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unit of analysis: one row = one (client_hash_id, content_hash_id, report_date) in
fact_content_daily_performance, filtered to month = 2026-03.


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

**Feature** (knowable before the decision moment, safe to use):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — daily performance, aggregated within
  the month up to the decision point.
- Derived: active-day count, average position, CTR, position volatility — all computed only
  from days already observed.

**Label / proxy** (what I'm predicting, never a feature):
- A within-month decline flag: did a page's impressions fall in the second half of the month
  versus the first half? This is a proxy for "this page's visibility is dropping," not a
  guarantee a refresh fixes it — same caveat as `w01`/`w02`.

**Context** (for grouping/joining/splitting, never fed to a model):
- `client_hash_id`, `content_hash_id` — pseudonyms, identify the row, never model inputs.
- `report_date`, `month` — used to build windows and filter, not raw inputs.

**Excluded** (with why):
- GA4 engagement columns — excluded from this pass because they're zero-filled with
  `ga4_data_available = FALSE` before a client's `ga4_data_start`; without filtering on that
  flag first, real missing history would look like real zero engagement.
- `fact_content_query_90d` — excluded from this notebook because its 90-day window overlaps
  the tail of the panel; joining it safely needs window-alignment work I'm not doing yet.

In [2]:
# Why GA4 columns are excluded right now — real counts for the month
ga4_check = con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1
""").df()
ga4_check



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,n
0,<NA>,3018741
1,False,6408671
2,True,413966


## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries prove the contract claims above. Then a five-feature frame for this lane, each
feature tagged with when it's actually knowable. Then the leakage trap: add one label-derived
column on purpose, watch the score jump, delete it, keep the honest number.

In [3]:
# Query 1 — GRAIN: does (client_hash_id, content_hash_id, report_date) really identify one row?
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate grain rows found: {len(grain_check)} (0 means the stated grain holds)")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0 (0 means the stated grain holds)


,client_hash_id,content_hash_id,report_date,n


In [4]:
# Query 2 — SLICE SIZE + DATE SPAN for this lane's slice
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
slice_stats

,row_count,n_clients,n_content,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [5]:
# Query 3 — AVAILABILITY: filter with IS TRUE, show how many rows survive
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
avail["pct_available"] = (avail["ga4_available_rows"] / avail["total_rows"] * 100).round(1)
avail


,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


**Five features, each tagged for availability:**
1. `total_impressions` — knowable at decision moment because it's a running sum of days already
   observed inside the month; no future day is touched.
2. `total_clicks` — same reason: only summed from report_dates on or before the decision point.
3. `avg_position` — averaged only over the observed days in the window, never a future ranking.
4. `active_days` — counts days with impressions > 0 within the same closed window, no lookahead.
5. `position_volatility` (STDDEV of avg_position) — computed over the same historical days,
   describing how noisy the page's ranking has been so far, not what it will be.

In [6]:
# Five features for Lane 2 (refresh/opportunity scoring), built from month = 2026-03 only.
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
""").df()

features["ctr"] = (features["total_clicks"] / features["total_impressions"]).round(4)

print(f"{len(features):,} content items, 5 core features each")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331,437 content items, 5 core features each


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,position_volatility,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,31,2.442255,0.0011
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,31,2.481312,0.0000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,31,1.243547,0.0011
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,31,0.983020,0.0026
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,21,22.609035,0.0000


**The trap.** Defining a within-month decline label — did impressions fall in the back half of
the month vs the front half — then deliberately adding one column derived straight from that
label (the raw percent change itself), fitting a one-line classifier, and watching the score
jump toward a suspiciously perfect number. Then deleting that column and keeping the honest
score.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

halves = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING imp_first_half > 0
""").df()

halves["pct_change"] = (halves["imp_second_half"] - halves["imp_first_half"]) / halves["imp_first_half"]
halves["is_declining"] = (halves["pct_change"] < -0.2).astype(int)

model_data = features.merge(
    halves[["client_hash_id", "content_hash_id", "is_declining", "pct_change"]],
    on=["client_hash_id", "content_hash_id"], how="inner"
)

honest_cols = ["total_impressions", "total_clicks", "avg_position", "active_days", "position_volatility"]

def quick_auc(cols, data):
    d = data.dropna(subset=cols + ["is_declining"])
    X, y = d[cols], d["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_cols, model_data)
print(f"Honest AUC (5 real features only): {honest_auc:.3f}")

leaked_cols = honest_cols + ["pct_change"]  # <- the trap: this IS the label, just rescaled
leaked_auc = quick_auc(leaked_cols, model_data)
print(f"Leaked AUC (adding pct_change, which the label is literally computed from): {leaked_auc:.3f}")

print("\npct_change deleted now — it only ever existed to demonstrate the leak.")
del model_data["pct_change"]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (5 real features only): 0.610
Leaked AUC (adding pct_change, which the label is literally computed from): 1.000

pct_change deleted now — it only ever existed to demonstrate the leak.


## 4. Data limits

**Named limitation:** this slice only captures a within-month, 31-day window — it can't say
anything about seasonal effects, year-over-year comparisons, or clients whose GSC history
doesn't even start until partway through `2026-03`. A client onboarded mid-month would show
artificially few `active_days` here, not because their pages are actually invisible, but
because there's no data for the days before onboarding — that's missing history, not zero
performance, exactly the trap the `flyrank-data` skill calls out.

In [8]:
onboarded_mid_month = con.sql(f"""
    SELECT COUNT(*) AS clients_starting_during_month
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start >= DATE '{MONTH}-01' AND gsc_data_start <= DATE '{MONTH}-31'
""").df()
onboarded_mid_month

,clients_starting_during_month
0,5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.